# Q1 — Vision Transformer (ViT) on CIFAR-10 (PyTorch)
This Colab-ready notebook implements a clean, educational ViT and trains it on CIFAR-10.
Run this on Colab with a GPU runtime.


In [2]:
# Install / imports (run on Colab)
!pip install -q torch torchvision timm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as T
import math, time, os
from tqdm import tqdm


error: incomplete-download

× Download failed after 6 attempts because not enough bytes were received (2.1 MB/73.6 MB)
╰─> URL: https://files.pythonhosted.org/packages/04/6e/650bb7f28f771af0cb791b02348db8b7f5f64f40f6829ee82aa6ce99aabe/torch-2.8.0-cp313-none-macosx_11_0_arm64.whl

note: This is an issue with network connectivity, not pip.
hint: Use --resume-retries to configure resume attempt limit.
error: incomplete-download

× Download failed after 6 attempts because not enough bytes were received (2.1 MB/73.6 MB)
╰─> URL: https://files.pythonhosted.org/packages/04/6e/650bb7f28f771af0cb791b02348db8b7f5f64f40f6829ee82aa6ce99aabe/torch-2.8.0-cp313-none-macosx_11_0_arm64.whl

note: This is an issue with network connectivity, not pip.
hint: Use --resume-retries to configure resume attempt limit.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


error: incomplete-download

× Download failed after 6 attempts because not enough bytes were received (2.1 MB/73.6 MB)
╰─> URL: https://files.pythonhosted.org/packages/04/6e/650bb7f28f771af0cb791b02348db8b7f5f64f40f6829ee82aa6ce99aabe/torch-2.8.0-cp313-none-macosx_11_0_arm64.whl

note: This is an issue with network connectivity, not pip.
hint: Use --resume-retries to configure resume attempt limit.
error: incomplete-download

× Download failed after 6 attempts because not enough bytes were received (2.1 MB/73.6 MB)
╰─> URL: https://files.pythonhosted.org/packages/04/6e/650bb7f28f771af0cb791b02348db8b7f5f64f40f6829ee82aa6ce99aabe/torch-2.8.0-cp313-none-macosx_11_0_arm64.whl

note: This is an issue with network connectivity, not pip.
hint: Use --resume-retries to configure resume attempt limit.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


ModuleNotFoundError: No module named 'torch'

In [ ]:
# Simple ViT implementation (educational)
import torch
import torch.nn as nn
import torch.nn.functional as F

class PatchEmbed(nn.Module):
    def __init__(self, img_size=32, patch_size=4, in_chans=3, embed_dim=256):
        super().__init__()
        self.patch_size = patch_size
        self.n_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        # x: B, C, H, W
        x = self.proj(x)  # B, embed_dim, H/ps, W/ps
        x = x.flatten(2).transpose(1,2)  # B, n_patches, embed_dim
        return x

class Attention(nn.Module):
    def __init__(self, dim, heads=8, qkv_bias=True, attn_drop=0., proj_drop=0.):
        super().__init__()
        self.heads = heads
        self.scale = (dim // heads) ** -0.5
        self.qkv = nn.Linear(dim, dim*3, bias=qkv_bias)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)
    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.heads, C // self.heads).permute(2,0,3,1,4)
        q, k, v = qkv[0], qkv[1], qkv[2]  # each: B, heads, N, head_dim
        attn = (q @ k.transpose(-2,-1)) * self.scale
        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)
        x = (attn @ v).transpose(1,2).reshape(B, N, C)
        x = self.proj(x)
        x = self.proj_drop(x)
        return x

class MLP(nn.Module):
    def __init__(self, in_features, hidden_features=None, drop=0.):
        super().__init__()
        hidden_features = hidden_features or in_features
        self.fc1 = nn.Linear(in_features, hidden_features)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden_features, in_features)
        self.drop = nn.Dropout(drop)
    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.drop(x)
        x = self.fc2(x)
        x = self.drop(x)
        return x

class Block(nn.Module):
    def __init__(self, dim, heads, mlp_ratio=4., qkv_bias=True, drop=0., attn_drop=0.):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = Attention(dim, heads=heads, qkv_bias=qkv_bias, attn_drop=attn_drop, proj_drop=drop)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = MLP(dim, int(dim*mlp_ratio), drop)
    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x

class ViT(nn.Module):
    def __init__(self, img_size=32, patch_size=4, in_chans=3, num_classes=10, embed_dim=256, depth=8, heads=8, mlp_ratio=4.):
        super().__init__()
        self.patch_embed = PatchEmbed(img_size, patch_size, in_chans, embed_dim)
        num_patches = self.patch_embed.n_patches
        self.cls_token = nn.Parameter(torch.zeros(1,1,embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches+1, embed_dim))
        self.pos_drop = nn.Dropout(0.)
        self.blocks = nn.ModuleList([
            Block(embed_dim, heads=heads, mlp_ratio=mlp_ratio) for _ in range(depth)
        ])
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)
        # init
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)
    def forward(self, x):
        B = x.shape[0]
        x = self.patch_embed(x)  # B, n_patches, embed_dim
        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat((cls_tokens, x), dim=1)
        x = x + self.pos_embed
        x = self.pos_drop(x)
        for blk in self.blocks:
            x = blk(x)
        x = self.norm(x)
        cls = x[:,0]
        out = self.head(cls)
        return out


In [ ]:
# Training + evaluation (small, Colab-friendly)
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from torchvision.datasets import CIFAR10
from torchvision.transforms import Compose, RandomHorizontalFlip, RandomCrop, ToTensor, Normalize

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Hyperparams (adjust as needed)
patch_size = 4
embed_dim = 256
depth = 8
heads = 8
mlp_ratio = 4.
batch_size = 128
epochs = 30  # increase to 60+ for better results
lr = 3e-4
weight_decay = 0.05

# Data
mean = (0.4914, 0.4822, 0.4465)
std = (0.2470, 0.2435, 0.2616)
train_transform = Compose([RandomCrop(32, padding=4), RandomHorizontalFlip(), ToTensor(), Normalize(mean, std)])
test_transform = Compose([ToTensor(), Normalize(mean, std)])
train_ds = CIFAR10(root='data', train=True, download=True, transform=train_transform)
test_ds = CIFAR10(root='data', train=False, download=True, transform=test_transform)
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False, num_workers=2, pin_memory=True)

model = ViT(img_size=32, patch_size=patch_size, in_chans=3, num_classes=10, embed_dim=embed_dim, depth=depth, heads=heads, mlp_ratio=mlp_ratio).to(device)
opt = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
sched = CosineAnnealingLR(opt, T_max=epochs)

best_acc = 0.0
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
    for x,y in pbar:
        x,y = x.to(device), y.to(device)
        opt.zero_grad()
        logits = model(x)
        loss = F.cross_entropy(logits, y)
        loss.backward()
        opt.step()
        running_loss += loss.item() * x.size(0)
    sched.step()
    # eval
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for x,y in test_loader:
            x,y = x.to(device), y.to(device)
            logits = model(x)
            preds = logits.argmax(dim=1)
            correct += (preds==y).sum().item()
            total += y.size(0)
    acc = correct/total*100
    print(f"Epoch {epoch+1}: Test Acc: {acc:.2f}%")
    if acc > best_acc:
        best_acc = acc
        torch.save(model.state_dict(), 'best_vit_cifar10.pth')
print("Best Test Acc: %.2f%%" % best_acc)
